# GraphVelo simulated vector-field benchmark

This notebook runs the GraphVelo embedding benchmark on the simulated vector-field datasets. The final plotting cell can be rerun to tune visualization parameters without retraining GraphVelo.


In [ ]:
import os

os.environ.setdefault("NUMBA_CACHE_DIR", "/private/tmp/numba_graphvelo_cache")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl_graphvelo_cache")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
os.makedirs(os.environ["NUMBA_CACHE_DIR"], exist_ok=True)
os.makedirs(os.environ["MPLCONFIGDIR"], exist_ok=True)

In [ ]:
import sys
import time
import warnings

import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import scvelo as scv
from sklearn.neighbors import NearestNeighbors

cwd = os.getcwd()
PROJECT_ROOT = cwd
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from embedding_metrics import evaluate_embedding_method
from graphvelo.graph_velocity import (
    GraphVelo,
    corr_kernel,
    cos_corr,
    density_corrected_transition_matrix,
    tangent_space_projection,
)

warnings.filterwarnings("ignore", category=FutureWarning)
scv.settings.verbosity = 1

In [ ]:
PATH_MAP = {
    "straight_line": "data/1d/straight_line.csv",
    "sine_curve": "data/1d/sine_curve.csv",
    "branch_2": "data/1d/branch_2.csv",
    "branch_4": "data/1d/branch_4.csv",
    "rotation": "data/2d/rotation.csv",
    "spiral": "data/2d/spiral.csv",
    "saddle": "data/2d/saddle.csv",
    "quadratic_source_sink": "data/2d/quadratic_source_sink.csv",
}

PLOT_ORDER = list(PATH_MAP)

NOISE = 0.3
EXTRA_DIM = 5
SEED = 42
N_NEIGHBORS = 30
MIN_DIST = 0.3
N_JOBS = 1
DATASETS = PLOT_ORDER

OUT_CSV = os.path.join(PROJECT_ROOT, "data/8_vf_collection/graphvelo.csv")
OUT_FIG = os.path.join(PROJECT_ROOT, "figures/8_vf_collection/graphvelo_embedding_streams.png")

In [ ]:
def load_vector_field(csv_path):
    df = pd.read_csv(csv_path)
    X = df[["x", "y"]].values
    V = df[["vx", "vy"]].values
    true_time = df["time"].values
    return X, V, true_time


def simulate_inputs(noise=NOISE, extra_dim=EXTRA_DIM, seed=SEED):
    rng = np.random.default_rng(seed)
    simulation_results = {}

    for name, path in PATH_MAP.items():
        X_gt, V_gt, true_time = load_vector_field(os.path.join(PROJECT_ROOT, path))
        X_noisy = X_gt + rng.normal(scale=noise, size=X_gt.shape)
        V_noisy = V_gt + rng.normal(scale=noise, size=V_gt.shape)
        X_dummy = rng.normal(scale=noise, size=(X_gt.shape[0], extra_dim))
        V_dummy = rng.normal(scale=noise, size=(V_gt.shape[0], extra_dim))

        simulation_results[name] = {
            "X": np.hstack([X_noisy, X_dummy]),
            "V": np.hstack([V_noisy, V_dummy]),
            "X_gt": X_gt,
            "V_gt": V_gt,
            "true_time": true_time,
        }

    return simulation_results


def fit_graphvelo_projection(X, V, n_neighbors=N_NEIGHBORS, min_dist=MIN_DIST, n_jobs=N_JOBS):
    adata = anndata.AnnData(X)
    adata.layers["position"] = X
    adata.layers["velocity"] = V

    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep="X")
    sc.tl.umap(adata, min_dist=min_dist, random_state=0)

    nbrs = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(X)
    adata.uns["neighbors"] = {"indices": nbrs.kneighbors(return_distance=False)[:, 1:]}

    gv = GraphVelo(
        adata,
        xkey="position",
        vkey="velocity",
        X_data=X,
        V_data=V,
        gene_subset=[f"feature_{i}" for i in range(X.shape[1])],
        approx=False,
    )

    transition = corr_kernel(
        gv.X,
        gv.V,
        gv.nbrs_idx,
        corr_func=cos_corr,
        softmax_adjusted=False,
    )
    transition = density_corrected_transition_matrix(transition).toarray()
    gv.T = sp.csr_matrix(tangent_space_projection(
        gv.X,
        gv.V,
        transition,
        gv.nbrs_idx,
        a=1,
        b=10,
        r=1,
        loss_func="log",
        n_jobs=n_jobs,
    ))

    X_emb = adata.obsm["X_umap"]
    V_emb = gv.project_velocity(X_emb)
    return X_emb, V_emb


def normalized_time(values):
    return (values - values.min()) / (values.max() - values.min() + 1e-12)

In [ ]:
simulation_results = simulate_inputs()
embedding_results = {}
records = []

for name in DATASETS:
    result = simulation_results[name]
    start = time.perf_counter()
    X_emb, V_emb = fit_graphvelo_projection(result["X"], result["V"])
    elapsed = time.perf_counter() - start

    embedding_results[name] = {
        "X_emb": X_emb,
        "V_emb": V_emb,
        "time": normalized_time(result["true_time"]),
    }

    metrics = evaluate_embedding_method(
        result["X_gt"],
        X_emb,
        result["V_gt"],
        V_emb,
        k=N_NEIGHBORS,
    )
    metrics["dataset"] = name
    metrics["runtime_seconds"] = elapsed
    records.append(metrics)
    print(f"{name}: {elapsed:.2f}s")

scores_df = pd.DataFrame(records).set_index("dataset")
scores_df.round(4)

In [ ]:
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
scores_df.to_csv(OUT_CSV)
OUT_CSV

## Tune the visualization

Rerun the next cell after changing `PLOT_KWARGS`. It reuses `embedding_results`, so it does not retrain GraphVelo.

In [ ]:
PLOT_KWARGS = {
    "density": 0.5,
    "arrow_size": 2.5,
    "linewidth": 3.0,
    "alpha": 0.3,
    "size": 800,
    "cmap": "viridis",
}

FIGSIZE_PER_PANEL = 4
SAVE_FIG = True


def panel_aspect(name):
    if name == "straight_line":
        return 2.5
    if name == "sine_curve":
        return 2.0
    if name in {"branch_2", "branch_4"}:
        return 1.5
    return "equal"


fig, axs = plt.subplots(1, len(DATASETS), figsize=(FIGSIZE_PER_PANEL * len(DATASETS), 4))
axs = np.atleast_1d(axs)

for ax, name in zip(axs, DATASETS):
    result = simulation_results[name]
    emb = embedding_results[name]

    plot_adata = anndata.AnnData(result["X"])
    plot_adata.obs["time"] = emb["time"]
    plot_adata.obsm["X_umap"] = emb["X_emb"]
    plot_adata.obsm["gv_umap"] = emb["V_emb"]

    scv.pl.velocity_embedding_stream(
        plot_adata,
        basis="umap",
        X=emb["X_emb"],
        V=emb["V_emb"],
        color="time",
        ax=ax,
        show=False,
        legend_loc=None,
        colorbar=False,
        title=None,
        **PLOT_KWARGS,
    )
    ax.set_aspect(panel_aspect(name))
    ax.axis("off")

plt.tight_layout()

if SAVE_FIG:
    os.makedirs(os.path.dirname(OUT_FIG), exist_ok=True)
    fig.savefig(OUT_FIG, dpi=300, bbox_inches="tight")
    print(f"Saved figure to {OUT_FIG}")

plt.show()